# SGLang 并发 16 凹陷 —— 查根因

## 已知什么

2026-09-05 三轮重复测量确认（`results_t4/sglang_outliers_repeat_2026-09-05.txt`）：

| 并发 | 三轮吞吐 tok/s | 中位 | 极差比 |
|---|---|---|---|
| 8 | 1171.1 / 1151.0 / 1121.6 | 1151.0 | 1.04× |
| **16** | **454.4 / 497.4 / 479.6** | **479.6** | **1.09×** |
| 32 | 919.5 / 883.8 / 924.4 | 919.5 | 1.05× |

**可复现（3/3），凹陷很深（只有并发 8 的 42%），但原因未查明。**
伴随特征：墙钟 16.5–18.0s（两侧 3–4s），TPOT p50 由 6.2ms 跳到 31–33ms。

## 四个实验（判据跑之前写死，不许事后改）

| # | 实验 | 若…… | 则…… |
|---|---|---|---|
| 0 | 并发细扫 8→32（含 10/12/14/18/20/24） | 凹陷是**窄坑**（只在 16 附近） | 指向某个跟 16 有关的阈值/边界 |
| | | 凹陷是**宽谷**（12–24 都塌） | 指向调度机制而非某个魔数 |
| 1 | 调度器日志切片 | 凹陷区 `#queue-req` 堆积、`#running-req` 震荡 | **排队/抢占**，不是解码慢 |
| | | 日志平稳无异常 | 排除调度层，往下查 kernel |
| 2 | 扫 `--chunked-prefill-size` | 凹陷**跟着参数移动或消失** | **根因锁定在 chunked prefill 边界** |
| | | 凹陷纹丝不动 | 与 chunked prefill 无关，排除 |
| 3 | 换回默认 flashinfer 后端 | 凹陷消失 | 是 **Triton 后端特有**（此前明确标注未测） |
| | | 凹陷仍在 | 与后端选择无关 |

**跑不出根因也算结果**——「排除了 A、B、C」本身就是诊断。四条都照实记。

> 环境沿用已验证口径：卸 `kernels`、卸 `torchaudio`、钉 `transformers==5.12.1`、不用 `sglang[all]`。


## 0. 环境

In [ ]:
import subprocess
out = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,compute_cap",
                      "--format=csv"], capture_output=True, text=True).stdout
print(out)
print("必须也是 Tesla T4，才与前几轮可比。")


## 1. 装 SGLang

In [ ]:
import importlib.metadata as md_, subprocess, sys, os, shutil

CHECK = ["sglang", "aiohttp", "torchvision"]
TF_PIN = "transformers==5.12.1"

def ver(p):
    try:
        return md_.version(p)
    except Exception:
        return None

def sh(cmd):
    return subprocess.run(cmd, shell=isinstance(cmd, str), capture_output=True, text=True)

missing = [p for p in CHECK if ver(p) is None]
print("缺失:", missing or "无")
if missing:
    if shutil.which("cargo") is None:
        sh("curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y --profile minimal")
    os.environ["PATH"] = os.path.expanduser("~/.cargo/bin") + ":" + os.environ["PATH"]
    print("装 sglang[srt] + aiohttp + torchvision + " + TF_PIN + "（约 5-10 分钟）...")
    r = sh([sys.executable, "-m", "pip", "install", "-q",
            "sglang[srt]", "aiohttp", "torchvision", TF_PIN])
    print("退出码:", r.returncode)
    if r.returncode:
        print(r.stdout[-2500:]); print(r.stderr[-2500:])
else:
    print("三个包都在，跳过安装。")

for pkg in ["kernels", "torchaudio"]:
    u = sh([sys.executable, "-m", "pip", "uninstall", "-y", "-q", pkg])
    print("  卸 %-12s 退出码 %s" % (pkg, u.returncode))
if ver("transformers") != "5.12.1":
    sh([sys.executable, "-m", "pip", "install", "-q", TF_PIN])
print()
for p in CHECK + ["transformers", "torch"]:
    print("  %-14s %s" % (p, ver(p) or "（未安装）"))


## 2. 探测参数 —— 先确认真实存在

`--chunked-prefill-size` 与 `--attention-backend` 的合法取值。
**若对不上，停下来把输出发我**，别猜参数名（这一格前几轮救过一次）。


In [ ]:
import subprocess, sys, re

r = subprocess.run([sys.executable, "-m", "sglang.launch_server", "--help"],
                   capture_output=True, text=True)
h = r.stdout + r.stderr
print("help returncode:", r.returncode, "| 长度:", len(h))
print()
for flag in ["--chunked-prefill-size", "--attention-backend", "--sampling-backend",
             "--max-running-requests", "--schedule-policy"]:
    print("  %-26s %s" % (flag, "存在" if flag in h else "不存在 !!"))
print()
m = re.search(re.escape("--attention-backend") + r"[^" + chr(10) + r"]*?\{([^}]*)\}", h)
print("attention-backend 取值:", m.group(1) if m else "（未列出）")
m2 = re.search(re.escape("--chunked-prefill-size") + r"[^" + chr(10) + r"]*", h)
print("chunked-prefill-size 行:", (m2.group(0)[:160] if m2 else "（未找到）"))


## 3. 写探针脚本

与 `bench_serving.py` **完全同样的请求形状**（同一批提示、max_tokens=128、temperature=0、stream），
只是并发列表可传。这样数字才能和前几轮并列。


In [ ]:
import io

P = []
P.append('# -*- coding: utf-8 -*-')
P.append('import argparse, asyncio, json, statistics as st, time')
P.append('import aiohttp')
P.append('URL = "http://127.0.0.1:8000/v1/chat/completions"')
P.append('MODEL = "Qwen/Qwen2.5-0.5B-Instruct"')
P.append('')
P.append('async def one(sess, prompt, max_tokens):')
P.append('    body = {"model": MODEL, "messages": [{"role":"user","content":prompt}],')
P.append('            "max_tokens": max_tokens, "temperature": 0.0, "stream": True}')
P.append('    t0 = time.perf_counter(); ttft=None; n=0; last=t0')
P.append('    async with sess.post(URL, json=body) as resp:')
P.append('        async for raw in resp.content:')
P.append('            line = raw.decode("utf-8").strip()')
P.append('            if not line.startswith("data: ") or line == "data: [DONE]": continue')
P.append('            d = json.loads(line[6:])["choices"][0].get("delta", {})')
P.append('            if d.get("content"):')
P.append('                now = time.perf_counter()')
P.append('                if ttft is None: ttft = now - t0')
P.append('                n += 1; last = now')
P.append('    return dict(ttft=ttft or 0.0, total=last-t0, n_tok=n)')
P.append('')
P.append('async def run_batch(n_conc, n_req, max_tokens):')
P.append('    prompts = ["Explain concept #%d in distributed systems." % i for i in range(n_req)]')
P.append('    sem = asyncio.Semaphore(n_conc)')
P.append('    async def guarded(sess, p):')
P.append('        async with sem: return await one(sess, p, max_tokens)')
P.append('    to = aiohttp.ClientTimeout(total=900)')
P.append('    async with aiohttp.ClientSession(timeout=to) as sess:')
P.append('        await one(sess, "warmup", 4)')
P.append('        t0 = time.perf_counter()')
P.append('        rs = await asyncio.gather(*(guarded(sess, p) for p in prompts))')
P.append('        wall = time.perf_counter() - t0')
P.append('    tot = sum(r["n_tok"] for r in rs)')
P.append('    tps = [(r["total"]-r["ttft"])/max(r["n_tok"]-1,1) for r in rs if r["n_tok"]>1]')
P.append('    return dict(conc=n_conc, wall=wall, tput=tot/wall,')
P.append('                ttft_p50=st.median(r["ttft"] for r in rs),')
P.append('                ttft_p99=sorted(r["ttft"] for r in rs)[int(len(rs)*0.99)-1],')
P.append('                tpot_p50=st.median(tps) if tps else 0.0)')
P.append('')
P.append('async def main(a):')
P.append('    concs = [int(x) for x in a.conc.split(",")]')
P.append('    out = []')
P.append('    print("%6s %7s %9s %12s %11s %11s %11s" % ("并发","请求","墙钟s","吞吐tok/s","TTFTp50","TTFTp99","TPOTp50"))')
P.append('    print("-"*72)')
P.append('    for c in concs:')
P.append('        n = max(c*4, 16)')
P.append('        r = await run_batch(c, n, a.max_tokens)')
P.append('        r["n_req"] = n')
P.append('        out.append(r)')
P.append('        print("%6d %7d %9.2f %12.1f %10.1fms %10.1fms %10.2fms" % (')
P.append('              c, n, r["wall"], r["tput"], r["ttft_p50"]*1e3, r["ttft_p99"]*1e3, r["tpot_p50"]*1e3))')
P.append('    json.dump(out, open(a.out, "w"))')
P.append('')
P.append('if __name__ == "__main__":')
P.append('    ap = argparse.ArgumentParser()')
P.append('    ap.add_argument("--conc", default="8,16,32")')
P.append('    ap.add_argument("--max-tokens", type=int, default=128)')
P.append('    ap.add_argument("--out", default="dip.json")')
P.append('    asyncio.run(main(ap.parse_args()))')

io.open("dip_probe.py", "w", encoding="utf-8").write(chr(10).join(P))
print("写出 dip_probe.py，", len(P), "行")


## 4. 启动器（带日志偏移记录）

每跑一档并发前后记一次日志字节偏移，之后就能把调度器日志**按并发切片**——
这是实验 1 的关键。


In [ ]:
import subprocess, sys, time, requests, os

MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
LOG = "/content/dip.log"

def serve(extra, tag, wait=420):
    subprocess.run(["pkill", "-f", "sglang.launch_server"], check=False)
    time.sleep(10)
    cmd = [sys.executable, "-m", "sglang.launch_server",
           "--model-path", MODEL, "--host", "127.0.0.1", "--port", "8000",
           "--context-length", "2048", "--mem-fraction-static", "0.80"] + extra
    print("启动:", " ".join(cmd[4:]))
    lg = open(LOG, "w")
    p = subprocess.Popen(cmd, stdout=lg, stderr=subprocess.STDOUT)
    for i in range(wait // 2):
        if p.poll() is not None:
            print("  [%s] 退出码 %s" % (tag, p.returncode))
            print(open(LOG).read()[-2500:])
            return None
        try:
            if requests.get("http://127.0.0.1:8000/v1/models", timeout=2).status_code == 200:
                print("  [%s] 就绪，用时 %ds" % (tag, i * 2))
                return p
        except requests.RequestException:
            pass
        time.sleep(2)
    print("  [%s] 超时" % tag)
    print(open(LOG).read()[-2500:])
    return None


def logsize():
    try:
        return os.path.getsize(LOG)
    except OSError:
        return 0


def logslice(a, b):
    with open(LOG, "rb") as f:
        f.seek(a)
        return f.read(max(0, b - a)).decode("utf-8", "replace")


## 5. 实验 0 —— 并发细扫，看凹陷是窄坑还是宽谷

默认后端（triton + pytorch），与前几轮一致。
逐档记录日志偏移，供实验 1 切片。


In [ ]:
import subprocess, sys, json, os

p = serve(["--attention-backend", "triton", "--sampling-backend", "pytorch"], "fine")
FINE, MARKS = None, {}
if p:
    CONCS = [8, 10, 12, 14, 16, 18, 20, 24, 32]
    print()
    print("%6s %7s %9s %12s %11s %11s %11s" % ("并发", "请求", "墙钟s", "吞吐tok/s", "TTFTp50", "TTFTp99", "TPOTp50"))
    print("-" * 72)
    FINE = []
    for c in CONCS:
        a = logsize()
        r = subprocess.run([sys.executable, "-u", "dip_probe.py",
                            "--conc", str(c), "--out", "one.json"],
                           capture_output=True, text=True)
        b = logsize()
        MARKS[c] = (a, b)
        if os.path.exists("one.json"):
            row = json.load(open("one.json"))[0]
            FINE.append(row)
            print("%6d %7d %9.2f %12.1f %10.1fms %10.1fms %10.2fms" % (
                  c, row["n_req"], row["wall"], row["tput"],
                  row["ttft_p50"] * 1e3, row["ttft_p99"] * 1e3, row["tpot_p50"] * 1e3))
        else:
            print("%6d  失败: %s" % (c, (r.stderr or r.stdout)[-300:]))
    json.dump(FINE, open("fine_sweep.json", "w"))


## 6. 实验 0 判定：窄坑还是宽谷

In [ ]:
if not FINE:
    print("没有细扫数据。")
else:
    tp = {r["conc"]: r["tput"] for r in FINE}
    base = max(tp.get(8, 0), tp.get(10, 0))
    print("以并发 8/10 的较大者为基准: %.1f tok/s" % base)
    print()
    print("%6s %12s %10s" % ("并发", "吞吐", "相对基准"))
    print("-" * 32)
    dipped = []
    for c in sorted(tp):
        ratio = tp[c] / base if base else 0
        flag = "  ← 塌" if ratio < 0.6 else ""
        print("%6d %12.1f %9.2f%s" % (c, tp[c], ratio, flag))
        if ratio < 0.6:
            dipped.append(c)
    print()
    print("塌陷档位:", dipped or "无")
    if not dipped:
        print("判定：本轮没复现凹陷 —— 与前三轮不一致，需先解释这个矛盾再往下。")
    elif len(dipped) == 1:
        print("判定：**窄坑**（只有并发 %d）。指向与该数值相关的阈值/边界。" % dipped[0])
    else:
        print("判定：**宽谷**（%s）。指向调度机制，而非某个魔数。" % dipped)


## 7. 实验 1 —— 调度器日志切片

看塌陷那一档的 `#running-req` / `#queue-req` / token usage 是否异常。


In [ ]:
import re

def summarize(c):
    if c not in MARKS:
        print("  并发 %d 没有日志区间" % c)
        return
    a, b = MARKS[c]
    seg = logslice(a, b)
    lines = [l for l in seg.splitlines() if re.search(r"#running-req|#queue-req|Decode batch|Prefill batch|preempt|Preempt", l)]
    print("  并发 %d：区间 %d 字节，匹配行 %d 条" % (c, b - a, len(lines)))
    run = [int(m.group(1)) for m in re.finditer(r"#running-req:\s*(\d+)", seg)]
    que = [int(m.group(1)) for m in re.finditer(r"#queue-req:\s*(\d+)", seg)]
    pre = len(re.findall(r"preempt", seg, re.I))
    if run:
        print("     #running-req  min/中位/max = %d / %d / %d" % (min(run), sorted(run)[len(run)//2], max(run)))
    if que:
        print("     #queue-req    min/中位/max = %d / %d / %d" % (min(que), sorted(que)[len(que)//2], max(que)))
    print("     出现 preempt 字样: %d 次" % pre)
    for l in lines[:4]:
        print("     |", l.strip()[:150])

for c in [8, 16, 32]:
    summarize(c)

print()
print("读法：若塌陷档的 #queue-req 明显堆高、#running-req 上不去或剧烈震荡、或出现 preempt，")
print("      则瓶颈在**排队/抢占**（与 TPOT 跳高一致）；若日志平稳，则排除调度层。")


## 8. 实验 2 —— 扫 `--chunked-prefill-size`

只测 8 / 16 / 32 三档（够判断凹陷在不在）。凹陷若跟着参数走，根因就锁定了。


In [ ]:
import subprocess, sys, json, os

CPS_RES = {}
for cps in ["512", "2048", "8192"]:
    print("=" * 56)
    print("chunked-prefill-size =", cps)
    pp = serve(["--attention-backend", "triton", "--sampling-backend", "pytorch",
                "--chunked-prefill-size", cps], "cps%s" % cps)
    if not pp:
        print("  起服务失败，跳过")
        continue
    r = subprocess.run([sys.executable, "-u", "dip_probe.py",
                        "--conc", "8,16,32", "--out", "cps.json"],
                       capture_output=True, text=True)
    print(r.stdout)
    if os.path.exists("cps.json"):
        CPS_RES[cps] = {row["conc"]: row["tput"] for row in json.load(open("cps.json"))}
print("=" * 56)
print("汇总:", json.dumps(CPS_RES, ensure_ascii=False))


## 9. 实验 2 判定

In [ ]:
if not CPS_RES:
    print("没有数据。")
else:
    print("%-10s %10s %10s %10s %12s" % ("cps", "并发8", "并发16", "并发32", "16/8"))
    print("-" * 56)
    for cps, d in CPS_RES.items():
        r8, r16, r32 = d.get(8), d.get(16), d.get(32)
        print("%-10s %10s %10s %10s %12s" % (
            cps,
            "%.1f" % r8 if r8 else "-",
            "%.1f" % r16 if r16 else "-",
            "%.1f" % r32 if r32 else "-",
            "%.2f" % (r16 / r8) if (r8 and r16) else "-"))
    print()
    ratios = {k: (d[16] / d[8]) for k, d in CPS_RES.items() if d.get(8) and d.get(16)}
    if len(ratios) >= 2:
        lo, hi = min(ratios.values()), max(ratios.values())
        print("16/8 比值区间: %.2f ~ %.2f" % (lo, hi))
        if hi > 0.75 and lo < 0.6:
            print("判定：**凹陷随 chunked-prefill-size 变化** —— 根因锁定在 chunked prefill。")
        elif hi < 0.6:
            print("判定：**凹陷纹丝不动** —— 与 chunked-prefill-size 无关，排除这条。")
        else:
            print("判定：不明确，需要更细的扫描。照实写。")


## 10. 实验 3 —— 换回默认 flashinfer 后端

此前明确标注过「没测默认后端」。flashinfer 在 sm_75 上可能起不来——
**起不来本身就是结果**，照实记，不要重试掩盖。


In [ ]:
import subprocess, sys, json, os

FI = None
pp = serve([], "flashinfer_default")   # 不传 backend 参数 = 用默认
if pp:
    r = subprocess.run([sys.executable, "-u", "dip_probe.py",
                        "--conc", "8,16,32", "--out", "fi.json"],
                       capture_output=True, text=True)
    print(r.stdout)
    if os.path.exists("fi.json"):
        FI = {row["conc"]: row["tput"] for row in json.load(open("fi.json"))}
        print("默认后端结果:", FI)
        if FI.get(8) and FI.get(16):
            ratio = FI[16] / FI[8]
            print("16/8 = %.2f" % ratio)
            print("判定：", "凹陷仍在 —— 与后端选择无关。" if ratio < 0.6
                  else "凹陷消失 —— 是 Triton 后端特有。")
else:
    print("默认后端起不来。**这本身是结果**：在 sm_75 上默认路径不可用，")
    print("这也解释了为什么前几轮必须显式指定 triton + pytorch。")
    print("实验 3 结论：无法在默认后端上验证凹陷，如实记为「未测到」。")


## 11. 边界

- 全部单实例、单卡 T4、0.5B 模型、2048 上下文。
- 实验 0 每档只跑一次（前面已确认凹陷可复现 3/3，这里重在**形状**而非精度）。
- 日志切片按时间区间归属，若服务端有异步刷盘，边界可能有几行串档。
- **若四个实验都没能锁定根因，就写「排除了 X、Y，未定位」——不要编解释。**
